# 🔎 MLflow Tracking — Experiment, Artifacts & Model Registry

1. 📋 Load experiment details by experiment name
2. 📦 Load artifacts for a run ID — list directories & files
3. 📝 Register the model in MLflow (version 1)
4. 🚀 Load the registered model by name & version
5. ✅ Test the model with sample data

---

In [8]:
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient

---
## 1️⃣ Load Experiment Details by Experiment Name

In [ ]:
EXPERIMENT_NAME = "RF model versioning demo"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print(f"Experiment Name : {experiment.name}")
print(f"Experiment ID   : {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Lifecycle Stage : {experiment.lifecycle_stage}")

In [ ]:
# List all runs in the experiment
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print(f"Total runs: {len(runs)}\n")
runs[['run_id', 'status', 'start_time']]

---
## 2️⃣ Load Artifacts for a Run ID — Display Directories & Files

In [ ]:
RUN_ID = "e92368539ff94b94a7f3d6356f2b3759"

client = MlflowClient()
artifacts = client.list_artifacts(RUN_ID)

print(f"Artifacts for Run ID: {RUN_ID}\n")

for artifact in artifacts:
    icon = "📁" if artifact.is_dir else "📄"
    print(f"{icon} {artifact.path}")
    # if directory, list its contents
    if artifact.is_dir:
        for sub in client.list_artifacts(RUN_ID, artifact.path):
            size = f"  ({sub.file_size} bytes)" if sub.file_size else ""
            print(f"   └── {sub.path}{size}")

---
## 3️⃣ Register the Model in MLflow (Version 1)

In [ ]:
MODEL_NAME = "rf_pipeline_model"
model_uri  = f"runs:/{RUN_ID}/best_rf_pipeline_model"

# Register — MLflow auto-assigns version 1 for a new model name
result = mlflow.register_model(model_uri, MODEL_NAME)

print(f"Model Name    : {result.name}")
print(f"Model Version : {result.version}")
print(f"Status        : {result.status}")

---
## 4️⃣ Load the Registered Model by Name & Version

In [ ]:
MODEL_VERSION = "1"

model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")

print(f"Loaded model  : {MODEL_NAME} v{MODEL_VERSION}")
print(f"Model type    : {type(model).__name__}")
print(f"Pipeline steps: {list(model.named_steps.keys())}")

---
## 5️⃣ Test the Model with Two Samples

In [9]:
samples = pd.DataFrame([
    {"Quantity": 6,  "UnitPrice": 2.55, "Country": "United Kingdom"},
    {"Quantity": 1,  "UnitPrice": 15.00, "Country": "Germany"}
])

predictions = model.predict(samples)

samples["Prediction"] = predictions
samples["Label"] = samples["Prediction"].map({0: "Low-Value", 1: "High-Value"})

samples

,Quantity,UnitPrice,Country,Prediction,Label
0,6,2.55,United Kingdom,1,High-Value
1,1,15.00,Germany,1,High-Value
